# Basic Imports & Common Settings

In [13]:
import os
import json

s_base_path = '/discover/nobackup/cmalings/output/ASIA-AQ/deepsensor_working_data'
s_path_scripts = '/discover/home/cmalings/code/scripts/deepsensor_scripts'
n_hours_for_job = 2

# Custom Settings

In [10]:
# basic settings
s_base_name_for_model = 'FullModel_wi_AK'
Consider_ground_monitors_as_input = True
Fraction_of_ground_monitors_to_train_on = 0.8
Use_GEOSCF = True
Use_TROPOMI = True
Use_GEMS = True
Use_GCAS = True
Use_LULC = True

# Crossvalidation Options
### Spatial CV: "randomly" split sites (according to site latitude) and uses all but one split for training, testing on the last split. If False, data from all sites are used for both training and testing.
Spatial_CV = True
n_fold_spatial_cv = 5

### Temporal CV: Hold out every [n_fold_temporal_cv]th timestamp for testing, use the rest for training
Temporal_CV = False
n_fold_temporal_cv = 7

### Split the data by month to define training and testing sets
Temporal_Month_Split = True
Train_in_March = True

# Create Folders and Write Settings JSONs

In [11]:
if not Spatial_CV:
    n_fold_spatial_cv = 1

if not Temporal_CV:
    n_fold_temporal_cv = 1

l_model_names = []

for i_fold_spatial_cv in range(n_fold_spatial_cv):
    for i_fold_temporal_cv in range(n_fold_temporal_cv):
        # Name the mode:
        s_spatial_cv = f'_scv{i_fold_spatial_cv}'
        if Temporal_CV:
            s_temporal_cv = f'_tcv{i_fold_temporal_cv}'
        elif Temporal_Month_Split:
            if Train_in_March:
                s_temporal_cv = '_tcvf'
            else:
                s_temporal_cv = '_tcvm'
        else:
            s_temporal_cv = ''
        s_name_for_model = s_base_name_for_model+s_spatial_cv+s_temporal_cv
        l_model_names += [s_name_for_model]
        
        # working data folder and file
        processor_dir = f"/discover/nobackup/cmalings/output/ASIA-AQ/deepsensor_working_data/{s_name_for_model}/"
        data_fpath = os.path.join(processor_dir,"tmp.nc")
        try:
            os.mkdir(processor_dir)
        except:
            pass
        
        # Write settings out to a JSON file
        d_settings = {'Model Name':s_name_for_model,
                      'Ground Monitors Considered for Input':Consider_ground_monitors_as_input,
                      'Ground Monitor Training Fraction':Fraction_of_ground_monitors_to_train_on,
                      'Use GEOS-CF':Use_GEOSCF,
                      'Use TROPOMI':Use_TROPOMI,
                      'Use GEMS':Use_GEMS,
                      'Use GCAS':Use_GCAS,
                      'Use LULC':Use_LULC,
                      'Spatial Cross-Validation':Spatial_CV,
                      'Number of Spatial Cross-Validation Folds':n_fold_spatial_cv,
                      'Spatial Cross-Validation Fold':i_fold_spatial_cv,
                      'Temporal Cross-Validation':Temporal_CV,
                      'Number of Temporal Cross-Validation Folds':n_fold_temporal_cv,
                      'Temporal Cross-Validation Fold':i_fold_temporal_cv,
                      'Temporal Split By Month':Temporal_Month_Split,
                      'Training in March':Train_in_March,
                     }
        with open(os.path.join(processor_dir,s_name_for_model+'_settings.json'), 'w') as o_file:
            json.dump(d_settings, o_file, indent=4)
            

# Create Batch Run Scripts

In [12]:
n_jobs = len(l_model_names)-1

s_code_shell = '''conda activate $HOME/.conda/envs/deepsensor_cenv
python $HOME/code/ASIAAQ_Deepsensor_Run.py {}
exit 0
'''
for i_model,s_model in enumerate(l_model_names):
    o_shell_file = open(os.path.join(s_path_scripts,s_base_name_for_model+'_shellscript_'+str(i_model)+'.sh'),'w')
    o_shell_file.write(s_code_shell.format(s_model))
    o_shell_file.close()

s_code_batch = '''#!/bin/bash
#SBATCH --job-name=Deepsensor_Array_Job
#SBATCH --partition=gpu_a100
#SBATCH --constraint=rome
#SBATCH --ntasks=12
#SBATCH --gres=gpu:1
#SBATCH --mem-per-gpu=122G
#SBATCH --time={n_hours_for_job}:00:00
#SBATCH -o output.%A_%a
#SBATCH -e error.%A_%a
#SBATCH --account=s1866
#SBATCH --array=0-{n_jobs}
chmod 777 $HOME/code/scripts/deepsensor_scripts/*
$HOME/code/scripts/deepsensor_scripts/{s_base_name_for_model}_shellscript_${SLURM_ARRAY_TASK_ID}.sh
exit 0
'''

o_batch_file = open(os.path.join(s_path_scripts,s_base_name_for_model+'_batch.sh'),'w')
o_batch_file.write(s_code_batch.format(n_hours_for_job=n_hours_for_job,
                                       n_jobs=n_jobs,
                                       s_base_name_for_model=s_base_name_for_model,
                                       SLURM_ARRAY_TASK_ID='{SLURM_ARRAY_TASK_ID}'))
o_batch_file.close()
